In [6]:
import xarray as xr
import torch

In [2]:
def post_processor(ds: xr.Dataset) -> xr.Dataset:
    """Converts the prediction output to an xarray dataset with the same dimensions/variables as input"""
    da = ds["__xarray_dataarray_variable__"]
    n_lev = 19
    variables = ["uo", "vo", "thetao", "so"]
    slices = [slice(i, i + n_lev) for i in range(0, len(variables) * n_lev, n_lev)]
    var_slices = {k: sl for k, sl in zip(variables, slices)}
    variables = {
        k: da.isel(var=sl).rename({"var": "lev"}) for k, sl in var_slices.items()
    }
    variables["zos"] = da.isel(var=-1).squeeze()
    return xr.Dataset(variables)


def prediction_data_test(ds_prediction: xr.Dataset, ds_input):
    """Testfunction to check post-processed prediction output for format"""
    # TODO: Run the test for the preprocessing data here and warn only if it fails
    # That data should have been checked before training and here we only strictly enforce that things reflect the state of the input data.

    expected_sizes = {"x": 360, "y": 180, "lev": 19}
    given_sizes = ds_prediction.sizes
    compare_dims = list(
        set(list(expected_sizes.keys()) + list(given_sizes.keys())) - set(["time"])
    )
    if any(expected_sizes[dim] != given_sizes[dim] for dim in compare_dims):
        raise ValueError(
            f"Input dataset does not have the right sizes. Expected{expected_sizes}, got {given_sizes}"
        )

    # ensure all dimensions have coordinate values
    dims_without_coords = [
        di for di in ds_prediction.dims if di not in ds_prediction.coords
    ]
    if len(dims_without_coords) > 0:
        raise ValueError(
            f"Found the following dimensions without coordinates: {dims_without_coords}"
        )

    # ensure the attributes are the same on both datasets
    if not ds_prediction.attrs == ds_input.attrs:
        raise ValueError(
            "Prediction and Input datasets do not have matching attributes"
        )

    # TODO: ensure that both arrays have the same coordinates


In [3]:
ds_input = xr.open_dataset("/vast/sd5313/data/m2lines/3D_ocean_data/OM4_Horizontal_Regrid_Old.zarr", engine='zarr', chunks={})
ds_prediction_raw = xr.open_dataset("/scratch/sd5313/M2Lines/emulator/Ocean_Emulator/Preds/ConvNext UNet Train3DSurfaceEval3D_Train_global_3D_Test_global_3D_all_N_train_4000_Lateral_Data_025_no_smooth/Pred_lateral_Fast_Data_025_global_3D_all_N_samples_4000_rand_seed_1.zarr", engine='zarr', chunks={})

# our groundtruth is always just a time slice of the training (training is a bad name
ds_groundtruth = ds_input.isel(time=slice(4141, 4741))
# quick fix: 
ds_prediction_raw = ds_prediction_raw.assign_coords(time=ds_groundtruth.time)
# ds_prediction_raw = ds_prediction_raw.swap_dims({'x':'y'}) # why does this not work?
ds_prediction_raw = ds_prediction_raw.rename({'x':'x_i', 'y':'y_i'}).rename({'x_i':'y', 'y_i':'x'})
ds_prediction = post_processor(ds_prediction_raw) #FIXME: This should be done before uploading the prediction output.
# quick fix (coords to lev)
ds_prediction = ds_prediction.assign_coords(lev=range(len(ds_prediction.lev)), x=ds_input.x, y=ds_input.y)

# Run the test to make sure the output is formatted correctly
prediction_data_test(ds_prediction, ds_input)

In [4]:
import numpy as np
ds_raw = xr.DataArray(np.random.random([3,180, 360, 77]), dims=['time', 'y','x', 'var'])

In [8]:
# quickly calculate area (TODO: This should be a coordinate on the ds_training)
import numpy as np

# area_weights = np.cos(np.deg2rad(ds_input.y))
area_weights = xr.open_dataset("/scratch/as15415/Data/CM2x_grids/Grid_New.nc").rename(
    {"dx": "dxu", "dy": "dyu"}
)["area_C"]

ds_prediction = ds_prediction.assign_coords(areacello=area_weights)
ds_input = ds_input.assign_coords(areacello=area_weights)
ds_groundtruth = ds_groundtruth.assign_coords(areacello=area_weights)

In [9]:
def profile_mean(ds:xr.Dataset) -> xr.Dataset:
    return ds.weighted(ds.areacello).mean(['x','y'])

def profile_std(ds:xr.Dataset) -> xr.Dataset:
    return ds.weighted(ds.areacello).std(['x','y'])

In [ ]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    profile_prediction = profile_mean(ds_prediction).load()
    profile_groundtruth = profile_mean(ds_groundtruth).load()
    profile_stdv_prediction = profile_std(ds_prediction).load()
    profile_stdv_groundtruth = profile_std(ds_groundtruth).load()

[#                                       ] | 3% Completed | 78m 23sss